Zad1

In [1]:
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"
NUM_FACTS = 20

def fetch_cat_fact(_=None):
    try:
        response = requests.get(CAT_API_URL)
        response.raise_for_status()
        return response.json().get('fact')
    except requests.RequestException:
        return None

def test_sequential():
    start_time = time.time()

    facts = []
    for _ in range(NUM_FACTS):
        facts.append(fetch_cat_fact())

    duration = time.time() - start_time
    print(f"Czas wykonania (sekwencyjnie): {duration:.4f} s")
    return duration

def test_concurrent():
    start_time = time.time()

    facts = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        results = executor.map(fetch_cat_fact, range(NUM_FACTS))
        facts = list(results)

    duration = time.time() - start_time
    print(f"Czas wykonania (wielowątkowo): {duration:.4f} s")
    return duration

if __name__ == "__main__":
    time_seq = test_sequential()
    print("-" * 50)

    time_conc = test_concurrent()
    print("-" * 50)

    speedup = time_seq / time_conc
    print(f"Podsumowanie:")
    print(f"Wielowątkowość była {speedup:.2f} razy szybsza od podejścia sekwencyjnego.")

Czas wykonania (sekwencyjnie): 3.1711 s
--------------------------------------------------
Czas wykonania (wielowątkowo): 0.5187 s
--------------------------------------------------
Podsumowanie:
Wielowątkowość była 6.11 razy szybsza od podejścia sekwencyjnego.


Zad2

In [2]:
import queue
import threading
import time

work_queue = queue.Queue()
MAX_NUMBERS = 10

def producer(q, limit):
    print("[Producent] Rozpoczynam produkcję...")
    for i in range(1, limit + 1):
        print(f"[Producent] Dodał do kolejki: {i}")
        q.put(i)
        time.sleep(0.1)

    q.put(None)
    q.put(None)
    print("[Producent] Zakończył pracę.")

def even_consumer(q):
    while True:
        item = q.get()

        if item is None:
            q.put(None)
            q.task_done()
            print("[Konsument 1 - Parzyste] Zakończył pracę.")
            break

        if item % 2 == 0:
            print(f"  -> [Konsument 1 - Parzyste] Przetwarza: {item}")
            time.sleep(0.15)
            q.task_done()
        else:
            q.put(item)
            q.task_done()
            time.sleep(0.05)

def odd_consumer(q):
    while True:
        item = q.get()

        if item is None:
            q.put(None)
            q.task_done()
            print("[Konsument 2 - Nieparzyste] Zakończył pracę.")
            break

        if item % 2 != 0:
            print(f"  -> [Konsument 2 - Nieparzyste] Przetwarza: {item}")
            time.sleep(0.15)
            q.task_done()
        else:
            q.put(item)
            q.task_done()
            time.sleep(0.05)

if __name__ == "__main__":
    t_producer = threading.Thread(target=producer, args=(work_queue, MAX_NUMBERS))
    t_consumer_even = threading.Thread(target=even_consumer, args=(work_queue,))
    t_consumer_odd = threading.Thread(target=odd_consumer, args=(work_queue,))

    t_consumer_even.start()
    t_consumer_odd.start()
    t_producer.start()

    t_producer.join()
    t_consumer_even.join()
    t_consumer_odd.join()

    print("-" * 50)
    print("Wszystkie zadania z kolejki zostały przetworzone. Program zakończył działanie.")

[Producent] Rozpoczynam produkcję...
[Producent] Dodał do kolejki: 1
  -> [Konsument 2 - Nieparzyste] Przetwarza: 1
[Producent] Dodał do kolejki: 2
  -> [Konsument 1 - Parzyste] Przetwarza: 2
[Producent] Dodał do kolejki: 3
  -> [Konsument 2 - Nieparzyste] Przetwarza: 3
[Producent] Dodał do kolejki: 4
  -> [Konsument 1 - Parzyste] Przetwarza: 4
[Producent] Dodał do kolejki: 5
  -> [Konsument 2 - Nieparzyste] Przetwarza: 5
[Producent] Dodał do kolejki: 6
  -> [Konsument 1 - Parzyste] Przetwarza: 6
[Producent] Dodał do kolejki: 7
  -> [Konsument 2 - Nieparzyste] Przetwarza: 7
[Producent] Dodał do kolejki: 8
  -> [Konsument 1 - Parzyste] Przetwarza: 8
[Producent] Dodał do kolejki: 9
  -> [Konsument 2 - Nieparzyste] Przetwarza: 9
[Producent] Dodał do kolejki: 10
  -> [Konsument 1 - Parzyste] Przetwarza: 10
[Producent] Zakończył pracę.
[Konsument 2 - Nieparzyste] Zakończył pracę.
[Konsument 1 - Parzyste] Zakończył pracę.
--------------------------------------------------
Wszystkie zadania z

Zad3

In [4]:
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    NUMBERS_RANGE = range(1, 10001)

    print("Rozpoczynam obliczenia sekwencyjne (dla porównania)...")
    start_time_seq = time.time()
    # Obliczamy wyniki standardowo - na jednym procesie
    results_seq = [calculate_power_sum(n) for n in NUMBERS_RANGE]
    duration_seq = time.time() - start_time_seq
    print(f"Czas wykonania (sekwencyjnie): {duration_seq:.4f} s")

    print("-" * 50)

    print("Rozpoczynam obliczenia wieloprocesowe...")
    start_time_mp = time.time()

    with multiprocessing.Pool() as pool:

        results_mp = pool.map(calculate_power_sum, NUMBERS_RANGE)

    duration_mp = time.time() - start_time_mp
    print(f"Czas wykonania (wieloprocesowo): {duration_mp:.4f} s")

    print("-" * 50)
    print("Podsumowanie:")
    print(f"Obliczono {len(results_mp)} wyników.")

    if duration_mp < duration_seq:
        speedup = duration_seq / duration_mp
        print(f"Przetwarzanie wieloprocesowe było {speedup:.2f} razy szybsze!")
    else:
        print("W tym konkretnym (bardzo małym) przypadku narzut na utworzenie procesów przewyższył zysk z nich.")

Rozpoczynam obliczenia sekwencyjne (dla porównania)...
Czas wykonania (sekwencyjnie): 0.6657 s
--------------------------------------------------
Rozpoczynam obliczenia wieloprocesowe...
Czas wykonania (wieloprocesowo): 0.7520 s
--------------------------------------------------
Podsumowanie:
Obliczono 10000 wyników.
W tym konkretnym (bardzo małym) przypadku narzut na utworzenie procesów przewyższył zysk z nich.
